# CODEX GBM (WangLab Visium HD) — HE cell-type preprocess

Join single-nuclei annotations (`cellID` / `cell_type` / `subcluster`) to microscope
pixel coordinates from `2_Single_Nuclei_Matrix/*_loc.csv`.

Writes under `data/CODEX/GBM/Results/`:
- `gbm_P174511_Initial_cell_info_HE_by_annotation.csv`
- `gbm_P179161_Recurrent_cell_info_HE_by_annotation.csv`

Coordinates in loc CSV are already microscope HE pixels (same canvas as StarDist).
Hierarchy: `3_Annotation_Table/GBM_sc_seg_celltypes_hierarchy.xlsx`.


In [3]:
## 2026.09.03 LLY: WangLab Visium HD Initial / Recurrent → Hist2Pheno
## Env: SeededNTM


In [4]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
GBM_DIR = REPO / "code" / "CODEX_gbm"
if str(GBM_DIR) not in sys.path:
    sys.path.insert(0, str(GBM_DIR))

from gbm_paths import (
    DEFAULT_RESULTS_DIR,
    DEFAULT_HIERARCHY_XLSX,
    annotation_xlsx_path,
    he_tif_path,
    load_gbm_celltype_hierarchy,
    loc_csv_path,
    normalize_gbm_cell_type,
    sample_ids,
    sample_config,
    stardist_csv_path,
)

DEFAULT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
hierarchy = load_gbm_celltype_hierarchy()
print("Hierarchy L2=", len(hierarchy), "from", DEFAULT_HIERARCHY_XLSX.name)
print(hierarchy.to_string(index=False))


Hierarchy L2= 16 from GBM_sc_seg_celltypes_hierarchy.xlsx
 celltype_level2 celltype_level1 celltype_level0
         AC-like           Tumor           Tumor
        MES-like           Tumor           Tumor
         OC-like           Tumor           Tumor
        NPC-like           Tumor           Tumor
             G1S           Tumor           Tumor
             G2M           Tumor           Tumor
         Mac_Tmr         Myeloid         Myeloid
        Mac_SPP1         Myeloid         Myeloid
       Mac_other         Myeloid         Myeloid
       Mac_SEPP1         Myeloid         Myeloid
      Lymphocyte           Lymph           Lymph
 Oligodendrocyte           Oligo           Oligo
        Vascular        Vascular        Vascular
             CAF        Vascular        Vascular
Collagen_fibrils        Vascular        Vascular
        lowQ_vas        Vascular        Vascular


## Join annotation + loc; write Results CSVs

In [5]:
def process_sample(sample: str, *, write: bool = True, plot: bool = True) -> pd.DataFrame:
    cfg = sample_config(sample)
    anno = pd.read_excel(annotation_xlsx_path(sample))
    loc = pd.read_csv(loc_csv_path(sample))
    need_anno = {"cellID", "cell_type", "subcluster"}
    miss = need_anno - set(anno.columns)
    if miss:
        raise KeyError(f"{sample} annotation missing {miss}")
    need_loc = {"id", "x", "y"}
    miss = need_loc - set(loc.columns)
    if miss:
        raise KeyError(f"{sample} loc missing {miss}")

    anno = anno.copy()
    anno["cell_id"] = anno["cellID"].astype(str).str.strip()
    anno["cell_type_raw"] = anno["cell_type"]
    anno["cell_type"] = anno["cell_type"].map(normalize_gbm_cell_type)
    # Empty / nan subcluster → LowQ (Ini has LowQ with null subcluster)
    sub = anno["subcluster"]
    anno["subcluster"] = sub.map(
        lambda v: "LowQ" if pd.isna(v) or str(v).strip() in ("", "nan") else str(v).strip()
    )

    loc = loc.copy()
    loc["cell_id"] = loc["id"].astype(str).str.strip()
    merged = anno.merge(loc[["cell_id", "x", "y"]], on="cell_id", how="inner", validate="one_to_one")
    merged["x_centroid"] = merged["x"].astype(float)
    merged["y_centroid"] = merged["y"].astype(float)
    merged["X_pix_HE"] = merged["x_centroid"]
    merged["Y_pix_HE"] = merged["y_centroid"]
    merged["sample"] = sample

    he = he_tif_path(sample)
    with tifffile.TiffFile(he) as tiff:
        h, w = tiff.pages[0].shape[:2]
    inside = (
        (merged["X_pix_HE"] >= 0) & (merged["X_pix_HE"] < w)
        & (merged["Y_pix_HE"] >= 0) & (merged["Y_pix_HE"] < h)
    )
    print(f"\n{sample}")
    print(f"  annotation n={len(anno):,}  loc n={len(loc):,}  joined={len(merged):,}")
    print(f"  HE TIFF {w}x{h}  inside={inside.sum():,}/{len(merged):,}")
    print(f"  cell_type: {merged['cell_type'].value_counts().to_dict()}")
    print(f"  subcluster: {merged['subcluster'].value_counts().to_dict()}")
    star = stardist_csv_path(sample)
    if star.is_file():
        star_n = sum(1 for _ in open(star)) - 1
        print(f"  StarDist CSV n={star_n:,}  ({star.name})")

    out_cols = [
        "cell_id", "sample", "cell_type", "cell_type_raw", "subcluster",
        "x_centroid", "y_centroid", "X_pix_HE", "Y_pix_HE",
    ]
    out = merged[out_cols].copy()
    out_path = Path(cfg["cell_info_csv"])
    if write:
        out.to_csv(out_path, index=False)
        print(f"  wrote {out_path}")

    if plot:
        fig, ax = plt.subplots(figsize=(10, 4))
        # downsample for QC
        rng = np.random.default_rng(0)
        idx = rng.choice(len(out), size=min(20000, len(out)), replace=False)
        sc = out.iloc[idx]
        for ct, g in sc.groupby("cell_type"):
            ax.scatter(g["X_pix_HE"], g["Y_pix_HE"], s=0.3, label=ct, alpha=0.6)
        ax.set_xlim(0, w); ax.set_ylim(h, 0)
        ax.set_aspect("equal")
        ax.set_title(f"{sample} nuclei on microscope HE ({w}x{h})")
        ax.legend(markerscale=6, fontsize=7, loc="upper right", frameon=False)
        fig.tight_layout()
        fig_dir = DEFAULT_RESULTS_DIR / "figures"
        fig_dir.mkdir(parents=True, exist_ok=True)
        fig_path = fig_dir / f"gbm_{sample}_celltype_on_HE_qc.png"
        fig.savefig(fig_path, dpi=150)
        plt.show()
        print(f"  QC fig {fig_path}")
    return out

for sample in sample_ids():
    process_sample(sample, write=True, plot=True)


KeyboardInterrupt: 